# 01. Data Exploration and Phase 1 Preparation

Notebook này thực hiện phase 1 theo đúng thứ tự: load dữ liệu, chuẩn hóa schema, EDA, k-core filtering, và temporal split.

## Mục tiêu

- Nạp hai file `body.tsv` và `title.tsv`.
- Chuẩn hóa kiểu dữ liệu và loại bỏ bản ghi lỗi.
- Tạo thống kê mô tả cho báo cáo phase 1.
- Lọc k-core để giảm nhiễu.
- Chia train/validation/test theo thời gian.

In [2]:
from pathlib import Path
import sys

import pandas as pd

root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent
sys.path.insert(0, str(root))

from src.phase1 import (
    apply_k_core_filter,
    combine_raw_datasets,
    summarize_hyperlinks,
    temporal_split,
)

body_path = root / 'data' / 'raw' / 'soc-redditHyperlinks-body.tsv'
title_path = root / 'data' / 'raw' / 'soc-redditHyperlinks-title.tsv'
interim_dir = root / 'data' / 'interim'
processed_dir = root / 'data' / 'processed'
interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
body_path, title_path

(WindowsPath('d:/Study/School/Social_Data_Analysis/Explainable_Signed_Link_Prediction_for_Reddit_Inter-Community_Conflict_Warning/data/raw/soc-redditHyperlinks-body.tsv'),
 WindowsPath('d:/Study/School/Social_Data_Analysis/Explainable_Signed_Link_Prediction_for_Reddit_Inter-Community_Conflict_Warning/data/raw/soc-redditHyperlinks-title.tsv'))

In [3]:
combined = combine_raw_datasets(body_path, title_path)
summary = summarize_hyperlinks(combined)
summary

{'rows': 858488,
 'unique_source_subreddits': 55863,
 'unique_target_subreddits': 34572,
 'timestamp_min': Timestamp('2013-12-31 16:20:20'),
 'timestamp_max': Timestamp('2017-04-30 16:58:21'),
 'sentiment_counts': {np.int64(1): 776278, np.int64(-1): 82210},
 'top_source_subreddits': source_subreddit
 subredditdrama      27636
 bestof              21170
 titlegore            9503
 shitredditsays       7839
 drama                6784
 shitpost             6658
 circlebroke2         6583
 switcharoo           6039
 shitamericanssay     5963
 hailcorporate        5360
 Name: count, dtype: Int64,
 'top_target_subreddits': target_subreddit
 askreddit        26622
 iama             13446
 pics             12578
 todayilearned    11124
 funny            10777
 videos           10013
 worldnews         9944
 news              7692
 politics          6114
 gaming            6097
 Name: count, dtype: Int64}

In [4]:
filtered = apply_k_core_filter(combined, k=5)
train_df, validation_df, test_df = temporal_split(filtered)

combined.to_csv(interim_dir / 'phase1_combined_clean.csv', index=False)
filtered.to_csv(processed_dir / 'phase1_kcore_filtered.csv', index=False)
train_df.to_csv(processed_dir / 'phase1_train.csv', index=False)
validation_df.to_csv(processed_dir / 'phase1_validation.csv', index=False)
test_df.to_csv(processed_dir / 'phase1_test.csv', index=False)

{
    'combined_rows': len(combined),
    'filtered_rows': len(filtered),
    'train_rows': len(train_df),
    'validation_rows': len(validation_df),
    'test_rows': len(test_df),
}

{'combined_rows': 858488,
 'filtered_rows': 708425,
 'train_rows': 619294,
 'validation_rows': 43048,
 'test_rows': 46083}

## Ghi chú

Khi notebook này chạy ổn, phase 1 sẽ được coi là hoàn tất và mình sẽ tick toàn bộ checklist phase 1 trước khi sang phase 2.